# Deebo Enjoys Encoding and Encryption
- Use a trie to encode and decode the code
- Use brute force to find multiplicative inverses

In [1]:
bits = '1111101010011111110001111101001111110111111100101111100111111101101111011111110100111110001111110011111111000111011111010011111100001111100111011101101111100011011111111111'
gold = {'decoded': 'deebruh likes warm toast.', 'cipher_map': {'a': 'z', 'b': ' ', 'c': 'l', 'd': 'g', 'e': 's', 'f': 'c', 'g': 'b', 'h': 'v', 'i': 'h', 'j': 'j', 'k': '.', 'l': 'e', 'm': 'n', 'n': 'f', 'o': 'x', 'p': ',', 'q': 'q', 'r': 'i', 's': 'r', 't': 'w', 'u': 'm', 'v': 'o', 'w': 'a', 'x': 'u', 'y': 't', 'z': 'd', ' ': 'p', ',': 'k', '.': 'y'}, 'ciphertext': 'gss imvpeh.srpazinpwxzrwy', 'encoded': '1111110011111110001111110001111110011111101111110000111111100111111101010111110010111111111111111100011111101001111110101110111111111011110111110000111111010111111100011111111100111111111011111010011111100011111110100'}

In [2]:
class Trie:
    def __init__( self, code ):
        self.ds = []
        self.code = code
        tmp = self.ds
        count = 0
        # Insert each character into the trie
        for char, encoding in code.items():
            for bit in encoding:
                # If the tree is not deep enough, deepen it
                if len( tmp ) < 2:
                    # A way to do multiple appends
                    tmp.extend( [ [], [] ] )
                # Go down the tree now that it has been deepened
                tmp = tmp[ int( bit ) ]
            # Once all bits have been exhausted, insert the character at the leaf
            tmp.append( char )
            # Reset the trie to the root node to insert the next character
            tmp = self.ds

    def decode( self, bits ):
        message = ""
        tmp = self.ds
        for bit in bits:
            # If code is not corrupted, will never run out of children
            tmp = tmp[ int( bit ) ]
            # If the current node has no children, it is a leaf
            if len( tmp ) < 2:
                message += tmp[ 0 ]
                # Reset the trie so that the next bits can be processed
                tmp = self.ds
        return message

    def encode( self, message ):
        return ''.join( [ self.code[ c ] for c in message ] )

In [3]:
# Brute force will work just fine
# Since there are only 29 elements
def find_inverse( e, base = 29 ):
    if e == 0:
        return 0
    for f in range( base ):
        if e * f % base == 1:
            return f

In [4]:
from code_table_lab5 import ALPHABET, CODE

trie = Trie( CODE )
my_decoded = trie.decode( bits )
num_chars = len( my_decoded )

# Map each character to its index to make the next step convenient
inv_ord_map = { c : i for i, c in enumerate( ALPHABET ) }

my_cipher_map = { c : ALPHABET[ ( find_inverse( inv_ord_map[ c ] ) + num_chars ) % 29 ] for c in ALPHABET }
my_ciphertext = ''.join( [ my_cipher_map[ c ] for c in my_decoded ] )
my_encoded = trie.encode( my_ciphertext )

my_summary = {
    "decoded" : my_decoded,
    "cipher_map" : my_cipher_map,
    "ciphertext" : my_ciphertext,
    "encoded" : my_encoded,
}

my_summary == gold

True

# Object Comprehension
- Store the object simply as raw data and shape separately
    - Eases several numerical computations
    - Even libraries like `numpy` employ the same trick
- For `__str__`, create a list-of-rows since it must be printed as such
    - Need starting point (in raw data list) and length of each row
    - Length of each row is present in shape
    - To calculate starting point, find cumulative sums of row lengths
- For `__add__` and `_mul_` simply need to create a new object by editing the raw data alone
- For `__radd__` and `__rmul__`, recall the following rules:
    - Upon encountering a binary operation `obj1 op obj2`, Python first attempts `obj1.op( obj2 )`
    - If that fails (e.g. exception due to incompatible datatype or not implemented error), Python then attempts `obj2.rop( obj1 )`
- For `T`, carefully place raw data in the slots in column-wise order

In [5]:
data = [1, 2, 3, 4, 5, 6]
shape = (3, 1, 2)
other_data = [0, 0, 0, 0, 0, 0]
other_shape = (3, 1, 2)
p, q, r, s = 0, 1, 1, 0
gold = {'as_string': '1 2 3\n4\n5 6', 'mixed_ops': '1 2 3\n4\n5 6', 'transpose': '1 4 6\n2\n3 5', 'array_add': '1 2 3\n4\n5 6', 'array_add_error': None}

In [6]:
class Jagged2DArray:
    
    def __init__( self, data, shape ):
        self.data = data
        self.shape = shape
        
    def __str__( self ):
        total = 0
        cum_sum = []
        for n in self.shape:
            cum_sum.append( total )
            total += n
        # Convert all data to string so that they can be "join"-ed
        # A list of integers cannot be joined
        # This could have been avoided by using a map operation
        str_data = [ str( n ) for n in self.data ]
        return '\n'.join( [ ' '.join( str_data[ start : start + length ] ) for start, length in zip( cum_sum, self.shape ) ] )

    def __add__( self, other ):
        if type( other ) == int:
            return Jagged2DArray( [ n + other for n in self.data ], self.shape )
        if type( other ) == Jagged2DArray:
            if other.shape != self.shape:
                raise ValueError( "Shapes must match for Jagged2DArray addition" )
            else:
                return Jagged2DArray( [ s + o for s, o in zip( self.data, other.data ) ], self.shape )

    def __radd__( self, other ):
        return self + other

    def __mul__( self, other ):
        if type( other ) == int:
            return Jagged2DArray( [ n * other for n in self.data ], self.shape )

    def __rmul__( self, other ):
        return self * other

    @property
    def T( self ):
        slots = [ [ 0 ] * cols_in_row for cols_in_row in self.shape ]
        num_cols = max( self.shape )
        num_rows = len( self.shape )
        filled_per_row = [ 0 ] * num_rows
        curr_idx = 0
        for col in range( num_cols ):
            for row in range( num_rows ):
                if filled_per_row[ row ] < self.shape[ row ]:
                    slots[ row ][ col ] = curr_idx
                    curr_idx += 1
                    filled_per_row[ row ] += 1
        permuted_idx = []
        # Flatten the list of lists
        [ permuted_idx.extend( row ) for row in slots ]
        return Jagged2DArray( [ self.data[ idx ] for idx in permuted_idx ], self.shape )

ja = Jagged2DArray( data, shape )
result = {
        "as_string": str( ja ),
        "mixed_ops": str( p + ( q * ja * r ) + s ),
        "transpose": str( ja.T ),
        "array_add": None,
        "array_add_error": None,
}
other = Jagged2DArray( other_data, other_shape )
try:
    result[ "array_add" ] = str( ja + other )
except ValueError as err:
    result[ "array_add_error" ] = str( err )

print( result )
print( str( ja.T ) )
jat = ja.T
print( jat.data )
result == gold

{'as_string': '1 2 3\n4\n5 6', 'mixed_ops': '1 2 3\n4\n5 6', 'transpose': '1 4 6\n2\n3 5', 'array_add': '1 2 3\n4\n5 6', 'array_add_error': None}
1 4 6
2
3 5
[1, 4, 6, 2, 3, 5]


True